<a href="https://colab.research.google.com/github/eshwarjyEnder-Cyber/Ender/blob/main/Encoder_Decoder(English_Hindi_Translator).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install datasets

In [2]:
import re
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


# -----------------------------
# Reproducibility
# -----------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# -----------------------------
# Device
# -----------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cpu


In [3]:
dataset = load_dataset(
    "cfilt/iitb-english-hindi"
)

print(dataset)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [4]:
for i in range(5):

    pair = dataset["train"][i]["translation"]

    print("English:", pair["en"])
    print("Hindi:  ", pair["hi"])
    print()

English: Give your application an accessibility workout
Hindi:   अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें

English: Accerciser Accessibility Explorer
Hindi:   एक्सेर्साइसर पहुंचनीयता अन्वेषक

English: The default plugin layout for the bottom panel
Hindi:   निचले पटल के लिए डिफोल्ट प्लग-इन खाका

English: The default plugin layout for the top panel
Hindi:   ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका

English: A list of plugins that are disabled by default
Hindi:   उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से निष्क्रिय किया गया है



In [6]:
def clean_sentence(sentence):

    sentence = sentence.lower().strip()

    # Separate punctuation
    sentence = re.sub(
        r"([?.!,])",
        r" \1 ",
        sentence
    )

    # Remove multiple spaces
    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    )

    return sentence

In [7]:
MAX_LENGTH = 12
N_PAIRS = 20000

pairs = []

for item in dataset["train"]:

    english = clean_sentence(
        item["translation"]["en"]
    )

    hindi = clean_sentence(
        item["translation"]["hi"]
    )

    if (
        len(english.split()) <= MAX_LENGTH
        and
        len(hindi.split()) <= MAX_LENGTH
        and
        len(english.split()) > 0
        and
        len(hindi.split()) > 0
    ):

        pairs.append(
            (english, hindi)
        )

    if len(pairs) == N_PAIRS:
        break


print(
    "Number of sentence pairs:",
    len(pairs)
)

Number of sentence pairs: 20000


In [8]:
for english, hindi in pairs[:10]:

    print("EN:", english)
    print("HI:", hindi)
    print()

EN: give your application an accessibility workout
HI: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें

EN: accerciser accessibility explorer
HI: एक्सेर्साइसर पहुंचनीयता अन्वेषक

EN: the default plugin layout for the bottom panel
HI: निचले पटल के लिए डिफोल्ट प्लग-इन खाका

EN: the default plugin layout for the top panel
HI: ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका

EN: a list of plugins that are disabled by default
HI: उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से निष्क्रिय किया गया है

EN: highlight duration
HI: अवधि को हाइलाइट रकें

EN: the duration of the highlight box when selecting accessible nodes
HI: पहुंचनीय आसंधि (नोड) को चुनते समय हाइलाइट बक्से की अवधि

EN: highlight border color
HI: सीमांत (बोर्डर) के रंग को हाइलाइट करें

EN: the color and opacity of the highlight border . 
HI: हाइलाइट किए गए सीमांत का रंग और अपारदर्शिता।

EN: highlight fill color
HI: भराई के रंग को हाइलाइट करें



In [9]:
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

In [10]:
def tokenize(sentence):

    return sentence.split()

In [11]:
def build_vocab(sentences, min_freq=2):

    word_frequency = {}

    for sentence in sentences:

        for word in tokenize(sentence):

            if word not in word_frequency:
                word_frequency[word] = 0

            word_frequency[word] += 1


    vocab = {

        PAD: 0,
        SOS: 1,
        EOS: 2,
        UNK: 3

    }


    for word, frequency in word_frequency.items():

        if frequency >= min_freq:

            vocab[word] = len(vocab)


    return vocab

In [12]:
english_sentences = [
    pair[0] for pair in pairs
]

hindi_sentences = [
    pair[1] for pair in pairs
]


en_vocab = build_vocab(
    english_sentences
)

hi_vocab = build_vocab(
    hindi_sentences
)


print(
    "English vocabulary:",
    len(en_vocab)
)

print(
    "Hindi vocabulary:",
    len(hi_vocab)
)

English vocabulary: 2005
Hindi vocabulary: 2108


In [13]:
en_itos = {
    index: word
    for word, index in en_vocab.items()
}

hi_itos = {
    index: word
    for word, index in hi_vocab.items()
}

In [14]:
def sentence_to_ids(
    sentence,
    vocab
):

    tokens = tokenize(sentence)

    ids = []

    # Start token
    ids.append(
        vocab[SOS]
    )

    for token in tokens:

        ids.append(
            vocab.get(
                token,
                vocab[UNK]
            )
        )

    # End token
    ids.append(
        vocab[EOS]
    )

    return ids

In [15]:
sentence = pairs[0][0]

print(sentence)

print(
    sentence_to_ids(
        sentence,
        en_vocab
    )
)

give your application an accessibility workout
[1, 4, 5, 6, 7, 8, 9, 2]


In [16]:
class TranslationDataset(Dataset):

    def __init__(
        self,
        pairs
    ):

        self.pairs = pairs


    def __len__(self):

        return len(self.pairs)


    def __getitem__(
        self,
        index
    ):

        english = self.pairs[index][0]

        hindi = self.pairs[index][1]


        english_ids = sentence_to_ids(
            english,
            en_vocab
        )

        hindi_ids = sentence_to_ids(
            hindi,
            hi_vocab
        )


        english_ids = torch.tensor(
            english_ids,
            dtype=torch.long
        )

        hindi_ids = torch.tensor(
            hindi_ids,
            dtype=torch.long
        )


        return (
            english_ids,
            hindi_ids
        )

In [17]:
def collate_fn(batch):

    english_sentences, hindi_sentences = zip(
        *batch
    )


    english_sentences = nn.utils.rnn.pad_sequence(

        english_sentences,

        batch_first=True,

        padding_value=en_vocab[PAD]

    )


    hindi_sentences = nn.utils.rnn.pad_sequence(

        hindi_sentences,

        batch_first=True,

        padding_value=hi_vocab[PAD]

    )


    return (
        english_sentences,
        hindi_sentences
    )

In [18]:
train_dataset = TranslationDataset(
    pairs
)


train_loader = DataLoader(

    train_dataset,

    batch_size=64,

    shuffle=True,

    collate_fn=collate_fn
)


print(
    "Number of batches:",
    len(train_loader)
)

Number of batches: 313


In [19]:
src, trg = next(
    iter(train_loader)
)

print("English batch shape:", src.shape)
print("Hindi batch shape:", trg.shape)

English batch shape: torch.Size([64, 14])
Hindi batch shape: torch.Size([64, 14])


In [20]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()


        self.embedding = nn.Embedding(

            input_dim,

            embedding_dim
        )


        self.rnn = nn.GRU(

            embedding_dim,

            hidden_dim,

            batch_first=True
        )


    def forward(self, src):

        # --------------------------------
        # src:
        #
        # [batch_size, source_length]
        # --------------------------------

        embedded = self.embedding(
            src
        )


        # --------------------------------
        # RNN reads the sentence
        # --------------------------------

        outputs, hidden = self.rnn(
            embedded
        )


        # hidden:
        #
        # [1, batch_size, hidden_dim]
        #
        # We only need the final state.

        context = hidden.squeeze(0)


        return context

In [21]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        context_dim,
        hidden_dim
    ):

        super().__init__()


        self.embedding = nn.Embedding(

            output_dim,

            embedding_dim
        )


        # --------------------------------
        # Decoder input =
        #
        # previous word embedding
        # +
        # encoder context
        # --------------------------------

        self.rnn = nn.GRUCell(

            embedding_dim + context_dim,

            hidden_dim
        )


        # --------------------------------
        # Predict next word using:
        #
        # decoder hidden state
        # +
        # encoder context
        # --------------------------------

        self.fc = nn.Linear(

            hidden_dim + context_dim,

            output_dim
        )


    def forward(
        self,
        input_token,
        hidden,
        context
    ):

        # --------------------------------
        # Previous generated word
        # --------------------------------

        embedded = self.embedding(
            input_token
        )


        # --------------------------------
        # Combine:
        #
        # E(y_t-1) + context
        # --------------------------------

        decoder_input = torch.cat(

            [
                embedded,
                context
            ],

            dim=1
        )


        # --------------------------------
        # Update decoder state
        #
        # s_t =
        # f(s_t-1, y_t-1, c)
        # --------------------------------

        hidden = self.rnn(

            decoder_input,

            hidden
        )


        # --------------------------------
        # Predict next word
        # --------------------------------

        prediction = self.fc(

            torch.cat(
                [
                    hidden,
                    context
                ],
                dim=1
            )
        )


        return (
            prediction,
            hidden
        )

In [22]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):

        super().__init__()

        self.encoder = encoder

        self.decoder = decoder


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = src.size(0)

        target_length = trg.size(1)

        output_dim = (
            self.decoder.fc.out_features
        )


        # Store all predictions

        outputs = torch.zeros(

            batch_size,

            target_length,

            output_dim

        ).to(device)


        # ==================================
        # ENCODER
        # ==================================

        context = self.encoder(
            src
        )


        # ==================================
        # Initial decoder state
        #
        # s_0 = h_T
        # ==================================

        hidden = context


        # ==================================
        # First decoder input
        #
        # <SOS>
        # ==================================

        input_token = trg[:, 0]


        # ==================================
        # DECODER
        # ==================================

        for t in range(
            1,
            target_length
        ):


            prediction, hidden = self.decoder(

                input_token,

                hidden,

                context

            )


            # Save prediction

            outputs[:, t, :] = prediction


            # ==================================
            # Teacher forcing
            # ==================================

            teacher_force = (

                random.random()
                < teacher_forcing_ratio

            )


            predicted_token = prediction.argmax(
                dim=1
            )


            if teacher_force:

                # Give decoder the actual
                # previous Hindi word

                input_token = trg[:, t]

            else:

                # Give decoder its own
                # previous prediction

                input_token = predicted_token


        return outputs

In [23]:
INPUT_DIM = len(en_vocab)

OUTPUT_DIM = len(hi_vocab)


ENC_EMBEDDING_DIM = 128

DEC_EMBEDDING_DIM = 128


ENCODER_HIDDEN_DIM = 256

DECODER_HIDDEN_DIM = 256

In [24]:
encoder = Encoder(

    input_dim=INPUT_DIM,

    embedding_dim=ENC_EMBEDDING_DIM,

    hidden_dim=ENCODER_HIDDEN_DIM
)


decoder = Decoder(

    output_dim=OUTPUT_DIM,

    embedding_dim=DEC_EMBEDDING_DIM,

    context_dim=ENCODER_HIDDEN_DIM,

    hidden_dim=DECODER_HIDDEN_DIM
)


model = Seq2Seq(

    encoder,

    decoder

).to(device)


print("Model created!")

Model created!


In [25]:
optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)


criterion = nn.CrossEntropyLoss(

    ignore_index=hi_vocab[PAD]

)

In [26]:
def train_one_epoch(
    model,
    train_loader
):

    model.train()

    total_loss = 0


    for src, trg in train_loader:

        src = src.to(device)

        trg = trg.to(device)


        optimizer.zero_grad()


        # --------------------------------
        # Run encoder + decoder
        # --------------------------------

        output = model(

            src,

            trg,

            teacher_forcing_ratio=0.5

        )


        # --------------------------------
        # Ignore <SOS>
        # --------------------------------

        output = output[:, 1:, :]

        trg = trg[:, 1:]


        output_dim = output.size(-1)


        # Flatten

        output = output.reshape(

            -1,

            output_dim

        )

        trg = trg.reshape(-1)


        # --------------------------------
        # Cross entropy
        # --------------------------------

        loss = criterion(

            output,

            trg

        )


        # --------------------------------
        # Backpropagation
        # --------------------------------

        loss.backward()


        # Prevent exploding gradients

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1

        )


        optimizer.step()


        total_loss += loss.item()


    return total_loss / len(train_loader)

In [27]:
N_EPOCHS = 5

loss_history = []


for epoch in range(
    N_EPOCHS
):

    loss = train_one_epoch(

        model,

        train_loader

    )


    loss_history.append(
        loss
    )


    print(

        f"Epoch {epoch+1:02d} | "
        f"Loss = {loss:.3f}"

    )

Epoch 01 | Loss = 3.885
Epoch 02 | Loss = 1.448
Epoch 03 | Loss = 0.627
Epoch 04 | Loss = 0.396
Epoch 05 | Loss = 0.322
